In [7]:
import numpy as np
import pandas as pd

root = '/home/qingyuyang/ESM4SL/data/hw3'

In [2]:
raw_train = pd.read_csv(f'{root}/sl_train.csv')
test = pd.read_csv(f'{root}/test.csv')

In [3]:
from sklearn.model_selection import train_test_split

train, val = train_test_split(raw_train, test_size=0.1, random_state=43, shuffle=True)

train.to_csv(f'{root}/train.csv', index=False)
val.to_csv(f'{root}/val.csv', index=False)

In [3]:
genes = list(set(raw_train['geneA_ID']) | set(raw_train['geneB_ID']) | set(test['geneA_ID']) | set(test['geneB_ID']))
len(genes)

9820

In [10]:
protein_info = pd.read_csv(f'{root}/protein_info.csv')

def form_id_seq_list(gids: set[int], mapping: pd.DataFrame, max_len: int = 2000) -> list[tuple[int, str]]:
    id_seq = []
    for gid in gids:
        print(gid)
        seq = mapping[mapping['NCBI_gene_id'] == gid]['Sequence'].values[0]
        if len(seq) > max_len:
            seq = seq[:max_len]
        id_seq.append((gid, seq))
    return id_seq

In [5]:
id_seq_list = form_id_seq_list(genes, protein_info)

In [6]:
import pickle

with open(f'{root}/all_id_seq.pkl', 'wb') as f:
    pickle.dump(id_seq_list, f)

Then, run esm2_gen.py

In [2]:
import os
general_esm2_list = [f.split('.')[0] for f in os.listdir('/home/qingyuyang/ESM4SL/data/hw3/ESM2_emb')]
slkb_esm2_list = [f.split('.')[0] for f in os.listdir('/home/qingyuyang/ESM4SL/data/SLKB/ESM2_emb')]
print(len(general_esm2_list), len(slkb_esm2_list))
len(set(slkb_esm2_list).difference(set(general_esm2_list)))

9820 5866


2083

In [13]:
import pandas as pd
test_gt = pd.read_csv('/home/qingyuyang/ESM4SL/data/hw3/sl_test.csv')
test_gene_set = set(test_gt['geneA_ID']) | set(test_gt['geneB_ID'])
have_emb_set = set([int(i.split('.')[0]) for i in os.listdir('/home/qingyuyang/ESM4SL/data/hw3/ESM2_emb')])
len(test_gene_set), len(test_gene_set.difference(have_emb_set))

(4191, 25)

In [14]:
new_test_gt = test_gt[test_gt['geneA_ID'].isin(have_emb_set) & test_gt['geneB_ID'].isin(have_emb_set)]
new_test_gt.to_csv('/home/qingyuyang/ESM4SL/data/hw3/test_new.csv', index=False)

In [16]:
train_df = pd.read_csv('/home/qingyuyang/ESM4SL/data/hw3/train.csv')

sl_pairs = [(row['geneA_ID'], row['geneB_ID'], row['label']) for _, row in train_df.iterrows()]
sl_pairs = np.array(sl_pairs, dtype=np.int32)

labels = sl_pairs[:, 2]
class_sample_count = np.array([len(np.where(labels == t)[0]) for t in np.unique(labels)])
weight = 1. / class_sample_count
samples_weight = np.array([weight[t] for t in labels])
print(len(samples_weight), set(samples_weight))

45651 {4.362621062734491e-05, 4.399665625412469e-05}


In [8]:
import random

all_pos_df = pd.read_csv('/home/qingyuyang/ESM4SL/data/pancancer/sl_synlethdb.csv')
all_pos_df.rename({'id_a': '0', 'id_b': '1', 'label': '2'}, axis=1, inplace=True)

gene_set = list(set(all_pos_df['0']) | set(all_pos_df['1']))
pos_position = np.asarray(all_pos_df[['0', '1']].values, dtype=int)

sl_pos_set = set(zip(pos_position[:, 0], pos_position[:, 1]))

neg_position = []
for ind_a in gene_set:
    for ind_b in gene_set:
        if (ind_a, ind_b) in sl_pos_set or (ind_b, ind_a) in sl_pos_set:
            continue
        if ind_a < ind_b:
            pair = (ind_a, ind_b)
            neg_position.append(pair)

In [9]:
sampled_neg = random.sample(neg_position, len(sl_pos_set))

sampled_neg = np.array(sampled_neg, dtype=int)
zeros_column = np.zeros((sampled_neg.shape[0], 1), dtype=int)
neg_arr = np.hstack((sampled_neg, zeros_column), dtype=int)
neg_df = pd.DataFrame(neg_arr, columns=['0', '1', '2'], dtype=int)

whole_df = pd.concat([all_pos_df, neg_df], ignore_index=True)
whole_df.to_csv('/home/qingyuyang/ESM4SL/data/pancancer/synlethdb_1:1.csv', index=False)

In [10]:
sampled_neg = random.sample(neg_position, len(sl_pos_set) * 5)

sampled_neg = np.array(sampled_neg, dtype=int)
zeros_column = np.zeros((sampled_neg.shape[0], 1), dtype=int)
neg_arr = np.hstack((sampled_neg, zeros_column), dtype=int)
neg_df = pd.DataFrame(neg_arr, columns=['0', '1', '2'], dtype=int)

whole_df = pd.concat([all_pos_df, neg_df], ignore_index=True)
whole_df.to_csv('/home/qingyuyang/ESM4SL/data/synlethdb_1:5.csv', index=False)

In [ ]:
base_df = pd.read_csv('/home/qingyuyang/ESM4SL/data/pancancer/synlethdb_1:1.csv')

In [ ]:
from utils import *

In [ ]:
def numpy2df(arr: np.ndarray):
    return pd.DataFrame(arr, columns=['0', '1', '2'])


class SplitPancancerDataset():
    def __init__(self, file_path, output_path):
        self.file_path = file_path
        self.output_path = output_path

    def SplitByPairs(self):
        df_sl = pd.read_csv(self.file_path)#, header=None

        sl_np_x = df_sl[['0', '1']].to_numpy()
        sl_np_y = df_sl['2'].to_numpy()

        # index is the fold
        index = 1

        random_seed = 43
        kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_seed)

        for train_index, test_index in kf.split(sl_np_x, sl_np_y):
            kf_train_valid = StratifiedKFold(n_splits=8, shuffle=True, random_state=random_seed)  # keep percentage of each class
            train_x = sl_np_x[train_index]
            train_y = sl_np_y[train_index]

            for train_train_index, train_valid_index in kf_train_valid.split(train_x, train_y):
                train_train_x = train_x[train_train_index]
                train_train_y = train_y[train_train_index].reshape(-1, 1)

                train_valid_x = train_x[train_valid_index]
                train_valid_y = train_y[train_valid_index].reshape(-1, 1)

                train_data = np.concatenate((train_train_x, train_train_y), axis=1)
                valid_data = np.concatenate((train_valid_x, train_valid_y), axis=1)
                break

            test_x = sl_np_x[test_index]
            test_y = sl_np_y[test_index].reshape(-1, 1)
            test_data = np.concatenate((test_x, test_y), axis=1)

            output_dir = f'{self.output_path}/C1/cv' + str(index)
            os.makedirs(output_dir, exist_ok=True)

            numpy2df(train_data).to_csv(f'{output_dir}/train.csv', index=False)
            numpy2df(valid_data).to_csv(f'{output_dir}/val.csv', index=False)
            numpy2df(test_data).to_csv(f'{output_dir}/test.csv', index=False)

            index += 1


    def get_pairs(self, sl_pairs, train_genes, test_genes, type):
        pairs_with_genes = []
        for pair in sl_pairs:
            if type == 1:
                if pair[0] in train_genes and pair[1] in train_genes:
                    pairs_with_genes.append(list(pair))
            elif type == 2:
                if (pair[0] in test_genes and pair[1] in train_genes) or (pair[0] in train_genes and pair[1] in test_genes):
                    pairs_with_genes.append(list(pair))
            elif type == 3:
                if pair[0] in test_genes and pair[1] in test_genes:
                    pairs_with_genes.append(list(pair))
        return pairs_with_genes


    def SplitByGene(self):
        df_sl = pd.read_csv(self.file_path) #, header=None

        genes = np.array(list(set(df_sl['0']) | set(df_sl['1'])))
        sl_pairs = df_sl.to_numpy()

        index = 1
        random_seed = 43
        kf = KFold(n_splits=5, shuffle=True, random_state=random_seed)

        for train_index, test_index in kf.split(genes):
            train_genes = genes[train_index]
            test_genes = genes[test_index]

            # split train/valid from train gene
            kf_train_valid = KFold(n_splits=8, shuffle=True, random_state=random_seed)
            for train_train_index, train_valid_index in kf_train_valid.split(train_genes):
                train_train_genes = train_genes[train_train_index]
                train_valid_genes = train_genes[train_valid_index]
                break
            
            train_data = self.get_pairs(sl_pairs, train_train_genes, test_genes=None, type=1)
            valid_c2_data = self.get_pairs(sl_pairs, train_train_genes, test_genes=train_valid_genes, type=2)
            valid_c3_data = self.get_pairs(sl_pairs, train_train_genes, test_genes=train_valid_genes, type=3)
            test_c2_data = self.get_pairs(sl_pairs, train_train_genes, test_genes, type=2)
            test_c3_data = self.get_pairs(sl_pairs, train_train_genes, test_genes, type=3)

            output_dir = f'{self.output_path}/C2/cv' + str(index)
            os.makedirs(output_dir, exist_ok=True)
            numpy2df(train_data).to_csv(f'{output_dir}/train.csv', index=False)
            numpy2df(valid_c2_data).to_csv(f'{output_dir}/val.csv', index=False)
            numpy2df(test_c2_data).to_csv(f'{output_dir}/test.csv', index=False)

            output_dir = f'{self.output_path}/C3/cv' + str(index)
            os.makedirs(output_dir, exist_ok=True)
            numpy2df(train_data).to_csv(f'{output_dir}/train.csv', index=False)
            numpy2df(valid_c3_data).to_csv(f'{output_dir}/val.csv', index=False)
            numpy2df(test_c3_data).to_csv(f'{output_dir}/test.csv', index=False)

            index += 1

In [22]:
split_cls = SplitPancancerDataset('/home/qingyuyang/ESM4SL/data/pancancer/synlethdb_1:1.csv',
                                  '/home/qingyuyang/ESM4SL/data/pancancer/synlethdb_1:1')
split_cls.SplitByPairs()
split_cls.SplitByGene()

In [23]:
split_cls = SplitPancancerDataset('/home/qingyuyang/ESM4SL/data/pancancer/synlethdb_1:5.csv',
                                  '/home/qingyuyang/ESM4SL/data/pancancer/synlethdb_1:5')
split_cls.SplitByPairs()
split_cls.SplitByGene()